In [1]:
import os, sys, platform; print(f"{platform.python_implementation()} {platform.python_version()} | {platform.system()} {platform.release()} | CPU: {platform.processor() or platform.machine()} | logical CPUs: {os.cpu_count()}")

CPython 3.12.13 | Windows 11 | CPU: Intel64 Family 6 Model 151 Stepping 2, GenuineIntel | logical CPUs: 24


In [2]:
from pathlib import Path

INPUT_PATH = Path(r"..\sample\PT\00013 Torso PET AC OSEM")
# INPUT_PATH = Path(r"..\sample\nema\Multiframe\CT\DISCIMG\IMAGES\CT0001")
INPUT_PATH


WindowsPath('../sample/PT/00013 Torso PET AC OSEM')

# Quick ITK timing codelets

Set `INPUT_PATH` to either a DICOM series directory or a single multiframe file.


In [3]:
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "bindings" / "python" / "dicomsdl").is_dir():
        return cwd
    if cwd.name == "tutorials" and (cwd.parent / "bindings" / "python" / "dicomsdl").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _find_repo_root()
repo_python_dir = REPO_ROOT / "bindings" / "python"
if repo_python_dir.is_dir():
    repo_python_text = str(repo_python_dir)
    if repo_python_text not in sys.path:
        sys.path.insert(0, repo_python_text)

INPUT_PATH = Path(INPUT_PATH)
if not INPUT_PATH.is_absolute():
    INPUT_PATH = (REPO_ROOT / INPUT_PATH).resolve()
if not INPUT_PATH.exists():
    raise FileNotFoundError(INPUT_PATH)

print(f"Repo root: {REPO_ROOT}")
print(f"Input path: {INPUT_PATH}")
print(f"Input kind: {'directory series' if INPUT_PATH.is_dir() else 'single file'}")


Repo root: C:\Lab\workspace\test.git
Input path: C:\Lab\workspace\sample\PT\00013 Torso PET AC OSEM
Input kind: directory series


## `dicomsdl.simpleitk_bridge.read_series_image`

In [4]:
from dicomsdl.simpleitk_bridge import read_series_image

def read_with_dicomsdl_simpleitk_bridge():
    return read_series_image(INPUT_PATH, to_modality_value=True)

%timeit read_with_dicomsdl_simpleitk_bridge()

118 ms ± 6.66 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## `SimpleITK`

In [5]:
import SimpleITK as sitk

if INPUT_PATH.is_dir():
    series_ids = list(sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(INPUT_PATH)) or [])
    if series_ids:
        file_names = list(sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(INPUT_PATH), series_ids[0]))
    else:
        file_names = list(sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(INPUT_PATH)))

def read_with_simpleitk_dir():
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(file_names)
    return reader.Execute()
    
def read_with_simpleitk_file():
    return sitk.ReadImage(INPUT_PATH)

if INPUT_PATH.is_dir():
    %timeit read_with_simpleitk_dir() # timeit with files in a directory
else:
    %timeit read_with_simpleitk_file() # timeit with a multiframe file

374 ms ± 49.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
